# Comprensión de los datos

**Proyecto:** RUES-SECOP  
**Metodología:** CRISP-DM  
**Fase:** 2. Comprensión de los datos

## Objetivo

Explorar las fuentes de información RUES y SECOP II para
comprender su estructura, contenido, cobertura y calidad.

A partir de esta exploración se identificarán las
posibilidades de análisis del tejido empresarial colombiano
y su relación con la contratación pública.

## Fuentes de información

- RUES: c82u-588k
- SECOP II - Contratos Electrónicos: jbjy-vk9h

## Actividades

1. Identificación de fuentes.
2. Exploración de columnas y tipos de datos.
3. Exploración de registros.
4. Análisis de cobertura temporal y geográfica.
5. Evaluación inicial de calidad.
6. Identificación de posibles relaciones entre fuentes.
7. Conclusiones de la comprensión de datos.

## Identificación de fuentes de datos

conectarnos al portal de datos abiertos de Colombia y obtener la información general de RUES y SECOP II.

In [0]:
# ============================================================
# 1. IDENTIFICACIÓN DE FUENTES DE DATOS
# ============================================================

import requests
import pandas as pd

# Fuentes oficiales
fuentes = {
    "RUES": "c82u-588k",
    "SECOP II": "jbjy-vk9h"
}

# Almacenamiento de resultados
resultados = []

# Consulta de metadatos
for nombre, identificador in fuentes.items():

    url = f"https://www.datos.gov.co/api/views/{identificador}.json"

    respuesta = requests.get(url, timeout=30)
    respuesta.raise_for_status()

    metadatos = respuesta.json()

    resultados.append({
        "Fuente": nombre,
        "Nombre del conjunto": metadatos.get("name"),
        "Identificador": identificador,
        "Número de columnas": len(metadatos.get("columns", [])),
        "Descripción": metadatos.get("description", "")
    })

# Presentación de resultados
df_fuentes = pd.DataFrame(resultados)

display(df_fuentes)

#### Resultados: identificación de fuentes

Se identificaron y consultaron satisfactoriamente dos fuentes
de datos del portal de datos abiertos de Colombia:

- **RUES:** contiene 36 columnas.
- **SECOP II:** contiene 95 columnas.

La consulta permitió recuperar los metadatos generales
de ambos conjuntos de datos sin descargar sus registros.

**Conclusión:** ambas fuentes están disponibles para
continuar con la exploración de su estructura y contenido.

**Siguiente actividad:** identificar las variables disponibles,
sus tipos de datos y sus posibles aplicaciones analíticas.

## Exploración de columnas y tipos de datos

In [0]:
# ============================================================
# 2. EXPLORACIÓN DE COLUMNAS Y TIPOS DE DATOS
# ============================================================

import requests
import pandas as pd

resultados_columnas = []

for fuente, identificador in fuentes.items():

    url = f"https://www.datos.gov.co/api/views/{identificador}.json"

    respuesta = requests.get(url, timeout=30)
    respuesta.raise_for_status()

    metadatos = respuesta.json()

    for columna in metadatos.get("columns", []):
        resultados_columnas.append({
            "Fuente": fuente,
            "Columna": columna.get("fieldName"),
            "Nombre descriptivo": columna.get("name"),
            "Tipo de dato": columna.get("dataTypeName"),
            "Descripción": columna.get("description", "")
        })

df_columnas = pd.DataFrame(resultados_columnas)

# Mostrar las columnas separadas por fuente
for fuente in fuentes:
    print(f"\nFUENTE: {fuente}")
    display(df_columnas[df_columnas["Fuente"] == fuente])

### 2.1. Agrupación de variables de RUES

La fuente RUES contiene 36 columnas que pueden clasificarse
en categorías según su propósito y significado.

| Categoría | Variables principales | Posibilidades de análisis |
|---|---|---|
| Identificación empresarial | `nit`, `matricula`, `razon_social`, `numero_identificacion` | Identificar empresas y evaluar posibles duplicados. |
| Registro mercantil | `camara_comercio`, `codigo_camara`, `inscripcion_proponente` | Analizar registros por cámara de comercio y estudiar la información de inscripción de proponentes. |
| Actividad económica | `cod_ciiu_act_econ_pri`, `cod_ciiu_act_econ_sec`, `ciiu3`, `ciiu4` | Caracterizar empresas por actividades económicas principales, secundarias y adicionales. |
| Naturaleza jurídica | `tipo_sociedad`, `organizacion_juridica`, `codigo_tipo_sociedad` | Clasificar empresas según su tipo de sociedad y organización jurídica. |
| Estado de matrícula | `estado_matricula`, `categoria_matricula`, `codigo_estado_matricula` | Analizar la distribución de registros según su estado y categoría. |
| Información temporal | `fecha_matricula`, `fecha_renovacion`, `fecha_cancelacion`, `fecha_vigencia`, `ultimo_ano_renovado`, `fecha_actualizacion` | Estudiar antigüedad, renovaciones, cancelaciones y actualización de registros. |
| Representación legal | `representante_legal`, `clase_identificacion_rl`, `num_identificacion_representante_legal` | Examinar la disponibilidad de información administrativa y de representación legal. |

### Hallazgos preliminares

1. **Clasificación económica:** RUES contiene códigos CIIU que permiten explorar la distribución empresarial por actividad económica.

2. **Estados de matrícula:** existen variables para distinguir los diferentes estados de los registros mercantiles, sin limitar el análisis únicamente a empresas activas.

3. **Información temporal:** se dispone de fechas relacionadas con matrícula, renovación, cancelación y actualización.

4. **Cobertura territorial:** no se identificaron columnas explícitas de departamento o municipio. Será necesario evaluar cómo relacionar las cámaras de comercio con información geográfica.

5. **Tipos de datos:** las 36 columnas están declaradas como `text`, por lo que será necesario examinar los formatos antes de realizar transformaciones.

### Conclusión

La estructura de RUES ofrece información relevante para caracterizar
el tejido empresarial colombiano según actividad económica,
naturaleza jurídica, estado de matrícula y características temporales.

Sin embargo, todavía debemos examinar los registros para comprobar
la cobertura, calidad y utilidad real de estas variables.

La siguiente actividad será analizar la estructura de SECOP II
para identificar sus posibilidades de análisis y los campos
que podrían permitir relacionarla con RUES.

## 2.2. Exploración estructural de SECOP II

### Objetivo

Identificar y clasificar las variables disponibles en SECOP II
para comprender qué información ofrece sobre la contratación
pública colombiana y evaluar sus posibles relaciones con RUES.

### Agrupación de variables

| Categoría | Variables principales | Posibilidades de análisis |
|---|---|---|
| Entidades contratantes | `nombre_entidad`, `nit_entidad`, `codigo_entidad`, `orden`, `sector`, `rama` | Identificar entidades públicas y analizar su participación en la contratación estatal. |
| Ubicación geográfica | `departamento`, `ciudad`, `localizaci_n`, `direcci_n_de_ejecuci_n_del_contrato` | Analizar la distribución territorial de entidades y lugares de ejecución contractual. |
| Identificación contractual | `proceso_de_compra`, `id_contrato`, `referencia_del_contrato`, `estado_contrato` | Identificar contratos, procesos de compra y estados contractuales. |
| Características contractuales | `tipo_de_contrato`, `modalidad_de_contratacion`, `objeto_del_contrato`, `descripcion_del_proceso`, `codigo_de_categoria_principal` | Analizar modalidades, objetos contractuales y categorías de bienes y servicios. |
| Proveedores | `documento_proveedor`, `tipodocproveedor`, `proveedor_adjudicado`, `codigo_proveedor`, `es_grupo`, `es_pyme` | Identificar proveedores adjudicados y estudiar su participación en la contratación pública. |
| Información económica | `valor_del_contrato`, `valor_facturado`, `valor_pagado`, `valor_pendiente_de_pago`, `valor_pendiente_de_ejecucion` | Analizar valores contractuales, facturación, pagos y saldos pendientes. |
| Fuentes de financiación | `origen_de_los_recursos`, `presupuesto_general_de_la_nacion_pgn`, `sistema_general_de_participaciones`, `sistema_general_de_regal_as`, `recursos_propios` | Examinar el origen presupuestal de los recursos contractuales. |
| Información temporal | `fecha_de_firma`, `fecha_de_inicio_del_contrato`, `fecha_de_fin_del_contrato`, `ultima_actualizacion`, `fecha_inicio_liquidacion` | Estudiar la contratación por periodos y las fechas de ejecución y liquidación. |
| Ejecución contractual | `liquidaci_n`, `dias_adicionados`, `duraci_n_del_contrato`, `el_contrato_puede_ser_prorrogado` | Examinar la ejecución, liquidación y modificaciones temporales de los contratos. |
| Sostenibilidad ambiental | `obligaci_n_ambiental`, `criterios_de_sostenibilidad_ambiental`, `requisitos_ambientales_en_las_especificaciones_t_cnicas`, `obligaciones_posconsumo` | Explorar criterios y obligaciones ambientales. |
| Información administrativa | `nombre_representante_legal`, `nombre_supervisor`, `nombre_ordenador_del_gasto`, `nombre_ordenador_de_pago` | Identificar información administrativa asociada a los contratos. |

### Hallazgos preliminares

1. **Distribución territorial:** SECOP II contiene variables de departamento y ciudad asociadas a las entidades contratantes. La ubicación de la entidad no necesariamente corresponde al lugar de ejecución contractual.

2. **Participación de proveedores:** existen identificadores que podrían permitir estudiar qué proveedores concentran mayor cantidad de contratos y valores adjudicados.

3. **Clasificación contractual:** la variable `codigo_de_categoria_principal` utiliza la clasificación UNSPSC para categorizar bienes y servicios contratados.

4. **Información económica:** se dispone de variables relacionadas con valores contractuales, pagos, facturación y saldos pendientes.

5. **Información temporal:** las fechas de firma, inicio y finalización permiten plantear análisis de contratación por periodos.

6. **Características empresariales:** la variable `es_pyme` podría permitir estudiar la participación de pequeñas y medianas empresas.

7. **Tipos de datos:** SECOP II presenta columnas declaradas como `text`, `number`, `calendar_date` y `url`.

### 2.3. Posibles relaciones entre RUES y SECOP II

| RUES | SECOP II | Posible relación |
|---|---|---|
| `nit` | `documento_proveedor` | Identificación tributaria de empresas proveedoras cuando el documento corresponda a un NIT. |
| `numero_identificacion` | `documento_proveedor` | Comparación de identificaciones según el tipo de documento. |
| `razon_social` | `proveedor_adjudicado` | Comparación de nombres empresariales como mecanismo complementario. |
| `cod_ciiu_act_econ_pri` | `codigo_de_categoria_principal` | Análisis entre actividades económicas CIIU y categorías contractuales UNSPSC, sujeto a una correspondencia válida. |

**Consideración:** las relaciones identificadas son preliminares.
Será necesario comprobar la compatibilidad, calidad y completitud
de los identificadores antes de integrar ambas fuentes.

### Conclusiones

La exploración estructural de SECOP II permitió identificar
95 columnas relacionadas con entidades públicas, proveedores,
contratos, valores económicos, fechas y características
administrativas.

La información disponible ofrece posibilidades para analizar
la distribución territorial de la contratación pública,
la concentración de contratos por proveedor, las modalidades
contractuales y los valores adjudicados.

La integración con RUES podría permitir estudiar las
actividades económicas de las empresas que contratan
con el Estado.

Sin embargo, estas posibilidades deberán verificarse
mediante la exploración de los registros reales,
la evaluación de calidad y el análisis de compatibilidad
entre ambas fuentes.

**Siguiente actividad:** exploración inicial del contenido
de RUES y SECOP II mediante muestras de registros.

## 3. Exploración inicial de los registros

### Objetivo

Examinar una muestra de registros de RUES y SECOP II para
comprender cómo se presenta la información dentro de cada
fuente y reconocer las características de sus valores.

### Actividades

1. Consultar una muestra inicial de registros de RUES.
2. Consultar una muestra inicial de registros de SECOP II.
3. Examinar los valores presentes en las columnas.
4. Identificar posibles valores nulos, formatos y particularidades.
5. Registrar los hallazgos relevantes de ambas fuentes.

### Consideraciones metodológicas

- Las consultas exploratorias utilizarán muestras pequeñas.
- No se aplicarán filtros que limiten el alcance del análisis definitivo.
- Los resultados de las muestras no se considerarán representativos
  de la totalidad de los datasets.
- No se realizarán transformaciones ni procesos de limpieza.
- No se crearán todavía tablas Bronze, Silver o Gold.

### Resultado esperado

Obtener una primera aproximación al contenido real de
ambas fuentes e identificar aspectos que requieran
una exploración más detallada.

Los hallazgos permitirán orientar las siguientes actividades
de comprensión de datos.

In [0]:
# ============================================================
# 3. EXPLORACIÓN INICIAL DE REGISTROS
# ============================================================

import requests
import pandas as pd

# Fuentes de datos
fuentes = {
    "RUES": "c82u-588k",
    "SECOP II": "jbjy-vk9h"
}

# Cantidad de registros por fuente
limite = 10

# Consultar y visualizar los registros
for nombre, identificador in fuentes.items():

    url = f"https://www.datos.gov.co/resource/{identificador}.json"

    respuesta = requests.get(
        url,
        params={"$limit": limite},
        timeout=30
    )
    respuesta.raise_for_status()

    registros = respuesta.json()

    df = pd.DataFrame(registros)

    print(f"\n{'=' * 60}")
    print(f"FUENTE: {nombre}")
    print(f"Registros obtenidos: {len(df)}")
    print(f"{'=' * 60}")

    display(df)

### 3.1. Exploración inicial de registros RUES

**Registros examinados:** 10  
**Método:** consulta API con límite de 10 registros, sin filtros.

#### Hallazgos

1. **Información histórica:** se observaron matrículas con fechas
   desde 1974 y registros cancelados en diferentes años.

2. **Estados de matrícula:** los diez registros examinados
   presentan el estado `CANCELADA`. Esta observación no permite
   establecer la distribución general de estados del dataset.

3. **Actividades económicas:** se encontraron códigos CIIU con
   diferentes formatos, incluidos valores numéricos y
   alfanuméricos, que requieren investigación adicional.

4. **Información temporal:** aparecen valores como `99991231`,
   `00000000` y `1900/01/01`, cuyo significado debe verificarse.

5. **Identificación empresarial:** se observaron diferencias entre
   los campos `nit` y `numero_identificacion`, lo que requiere
   evaluar su consistencia y significado.

6. **Completitud:** algunas variables contienen valores nulos,
   especialmente en información complementaria de personas
   naturales y representantes legales.

7. **Cobertura territorial:** se identificaron registros asociados
   a diferentes cámaras de comercio.

#### Conclusión

La muestra inicial evidencia que RUES contiene información
empresarial histórica y registros con diferentes características.

También revela posibles desafíos relacionados con formatos,
valores especiales, identificadores y completitud de variables.

Estos hallazgos son exploratorios y deberán comprobarse
mediante consultas de mayor cobertura antes de establecer
conclusiones sobre la totalidad del dataset.

### 3.2. Exploración inicial de registros SECOP II

**Registros examinados:** 10  
**Método:** consulta API con límite de 10 registros, sin filtros.

#### Hallazgos

1. **Cobertura territorial:** se observaron contratos de entidades
   registradas en Antioquia, Bogotá, Santander, Bolívar y Tolima.

2. **Información temporal:** las fechas de firma de los contratos
   examinados se encuentran entre 2020 y 2026.

3. **Tipos de contrato:** los diez registros corresponden a
   prestación de servicios. Esto no representa necesariamente
   la distribución general de tipos contractuales.

4. **Modalidades de contratación:** se identificaron contratos
   bajo contratación directa y régimen especial.

5. **Proveedores:** los registros examinados corresponden a
   personas naturales identificadas mediante cédula de ciudadanía.

6. **Información económica:** se observaron valores contractuales
   entre $1.877.120 y $33.600.000 COP.

7. **Estados contractuales:** aparecen diferentes estados,
   como En ejecución, Cerrado, Modificado y terminado.

8. **Completitud:** se identificaron valores como No definido,
   Sin Descripcion y null, que requieren evaluación posterior.

9. **Consistencia temporal:** algunos contratos presentan fechas
   de finalización anteriores a la fecha actual, pero mantienen
   el estado En ejecución. Se requiere investigar su significado
   y actualización.

#### Conclusión

La muestra inicial de SECOP II evidencia la disponibilidad
de información territorial, económica, temporal y contractual.

También demuestra que la contratación pública incluye
proveedores que son personas naturales, por lo que el análisis
no debe limitarse automáticamente a sociedades comerciales.

Se identificaron aspectos que requieren evaluación adicional,
especialmente la completitud de variables, la normalización
de categorías y la interpretación de estados contractuales.

Estos hallazgos son preliminares y no permiten establecer
tendencias generales sobre la contratación pública colombiana.

## 4. Cobertura y distribución de los datos

### Objetivo

Determinar el volumen de registros y la cobertura temporal
y territorial de RUES y SECOP II, para comprender el alcance
de la información disponible.

### Actividades

1. Consultar el número total de registros de cada fuente.
2. Identificar los periodos temporales disponibles.
3. Examinar la distribución de registros por territorio.
4. Identificar posibles limitaciones de cobertura.

### Consideraciones

- Se utilizarán consultas agregadas mediante las API.
- No se descargarán los datasets completos.
- No se excluirán registros por estado, fecha o actividad económica.
- Se conservará la distinción entre cámaras de comercio
  de RUES y departamentos de las entidades de SECOP II.

### Resultado esperado

Obtener indicadores que permitan dimensionar ambas fuentes
y orientar las siguientes actividades de exploración.

In [0]:
# ============================================================
# 4.1. VOLUMEN TOTAL DE REGISTROS
# ============================================================

import requests
import pandas as pd

fuentes = {
    "RUES": "c82u-588k",
    "SECOP II": "jbjy-vk9h"
}

resultados = []

for nombre, identificador in fuentes.items():

    url = f"https://www.datos.gov.co/resource/{identificador}.json"

    respuesta = requests.get(
        url,
        params={"$select": "count(*) as total"},
        timeout=180
    )

    respuesta.raise_for_status()

    datos = respuesta.json()

    resultados.append({
        "Fuente": nombre,
        "Total de registros": int(datos[0]["total"])
    })

df_volumen = pd.DataFrame(resultados)

display(df_volumen)

### 4.1. Resultados del volumen de registros

| Fuente | Total de registros |
|---|---:|
| RUES | 9.466.228 |
| SECOP II | 5.687.382 |
| **Total combinado** | **15.153.610** |

#### Hallazgos

1. RUES contiene un volumen superior a nueve millones
   de registros de información mercantil.

2. SECOP II contiene más de cinco millones de registros
   relacionados con contratación pública.

3. Ambas fuentes representan unidades de información
   diferentes, por lo que sus cantidades no deben
   interpretarse como un número de entidades únicas.

4. Los conteos corresponden al momento de ejecución
   de las consultas y pueden cambiar con las
   actualizaciones de las fuentes.

#### Conclusión

El volumen disponible justifica utilizar consultas
agregadas para la exploración inicial, evitando
descargas completas innecesarias.

Antes de definir la estrategia de ingesta será necesario
estudiar la cobertura temporal, territorial y la
granularidad de los registros.

### 4.2: cobertura temporal de RUES y SECOP II.

In [0]:

import requests
import pandas as pd

consultas = {
    "RUES": {
        "id": "c82u-588k",
        "campo": "fecha_matricula",
        "expresion": "fecha_matricula"
    },
    "SECOP II": {
        "id": "jbjy-vk9h",
        "campo": "fecha_de_firma",
        "expresion": "fecha_de_firma"
    }
}

resultados = []

for fuente, configuracion in consultas.items():

    url = (
        f"https://www.datos.gov.co/resource/"
        f"{configuracion['id']}.json"
    )

    campo = configuracion["expresion"]

    parametros = {
        "$select": (
            f"min({campo}) as fecha_minima, "
            f"max({campo}) as fecha_maxima, "
            f"count({campo}) as registros_con_fecha"
        )
    }

    respuesta = requests.get(
        url,
        params=parametros,
        timeout=120
    )

    respuesta.raise_for_status()
    datos = respuesta.json()[0]

    resultados.append({
        "Fuente": fuente,
        "Campo": configuracion["campo"],
        "Fecha mínima": datos.get("fecha_minima"),
        "Fecha máxima": datos.get("fecha_maxima"),
        "Registros con fecha": int(
            datos.get("registros_con_fecha", 0)
        )
    })

df_cobertura_temporal = pd.DataFrame(resultados)

display(df_cobertura_temporal)

### identificar fechas válidas en RUES

In [0]:

# 4.2.1. VALIDACIÓN EXPLORATORIA DE FECHAS RUES

import requests
import pandas as pd

url = "https://www.datos.gov.co/resource/c82u-588k.json"

consultas = {
    "Fechas 00000000": "fecha_matricula = '00000000'",
    "Fechas 99999999": "fecha_matricula = '99999999'",
    "Fechas 99991231": "fecha_matricula = '99991231'"
}

resultados = []

for descripcion, condicion in consultas.items():

    respuesta = requests.get(
        url,
        params={
            "$select": "count(*) as total",
            "$where": condicion
        },
        timeout=120
    )

    respuesta.raise_for_status()

    resultados.append({
        "Categoría": descripcion,
        "Registros": int(respuesta.json()[0]["total"])
    })

display(pd.DataFrame(resultados))


# Fechas con formato básico AAAAMMDD, en un rango
# razonable para la exploración inicial.
parametros = {
    "$select": (
        "min(fecha_matricula) as fecha_minima, "
        "max(fecha_matricula) as fecha_maxima, "
        "count(*) as total"
    ),
    "$where": (
        "fecha_matricula between '18000101' and '20261231' "
        "AND fecha_matricula like '________'"
    )
}

respuesta = requests.get(
    url,
    params=parametros,
    timeout=120
)

respuesta.raise_for_status()

print("Intervalo preliminar de fechas RUES:")
display(pd.DataFrame(respuesta.json()))

### 4.3. Resultados de cobertura temporal

#### RUES

- Total de registros: 9.466.228.
- Registros con valor en fecha_matricula: 9.437.067.
- Registros con fecha 00000000: 19.801.
- Registros con fecha 99999999: 1.
- Registros con fecha 99991231: 0.
- Registros dentro del rango exploratorio: 9.417.139.
- Fecha mínima encontrada: 1878-11-16.
- Fecha máxima encontrada: 2026-11-06.

**Hallazgos:**

RUES contiene información de matrículas mercantiles
correspondientes a diferentes periodos históricos.

Se identificaron valores especiales que no representan
fechas válidas y requieren tratamiento posterior.

El intervalo identificado es preliminar, dado que
todavía no se ha comprobado la validez calendárica
de todos los valores.

#### SECOP II

- Total de registros: 5.687.382.
- Registros con fecha de firma: 5.620.281.
- Fecha mínima de firma: 2015-06-11.
- Fecha máxima de firma: 2026-11-08.

**Hallazgos:**

SECOP II presenta registros con fechas de firma
comprendidas entre junio de 2015 y noviembre de 2026.

La cobertura temporal observada es diferente a la
de RUES, aspecto que deberá considerarse al estudiar
posibles relaciones entre ambas fuentes.

#### Conclusión general

Las fuentes presentan coberturas temporales diferentes.

RUES permite explorar información mercantil histórica,
mientras que SECOP II proporciona información contractual
con fechas de firma desde 2015.

La identificación de estos intervalos no implica
restringir el análisis a un periodo específico.

La validez de las fechas, la distribución anual y la
compatibilidad temporal entre fuentes deberán evaluarse
en las siguientes actividades.

### Distribución anual de RUES

In [0]:
# ============================================================
# 4.3.1. DISTRIBUCIÓN ANUAL DE RUES
# ============================================================

import requests
import pandas as pd

url = "https://www.datos.gov.co/resource/c82u-588k.json"

resultados = []

# Rango exploratorio identificado anteriormente
for anio in range(1878, 2027):

    parametros = {
        "$select": "count(*) as total",
        "$where": (
            f"fecha_matricula >= '{anio}0101' "
            f"AND fecha_matricula <= '{anio}1231' "
            "AND fecha_matricula like '________'"
        )
    }

    respuesta = requests.get(
        url,
        params=parametros,
        timeout=120
    )

    if not respuesta.ok:
        print(respuesta.text[:1000])

    respuesta.raise_for_status()

    total = int(respuesta.json()[0]["total"])

    resultados.append({
        "anio": anio,
        "total_registros": total
    })

df_anual_rues = pd.DataFrame(resultados)

# Mostrar únicamente años con registros
display(
    df_anual_rues[
        df_anual_rues["total_registros"] > 0
    ]
)

print(
    "Total de registros distribuidos:",
    df_anual_rues["total_registros"].sum()
)

#### Verificación de consistencia

- Registros identificados en el rango exploratorio: 9.417.139.
- Registros distribuidos por año: 9.417.138.
- Diferencia: 1 registro.

La distribución anual presenta una consistencia
prácticamente completa con el conteo exploratorio previo.

La diferencia de un registro podría estar relacionada
con actualizaciones de la fuente entre consultas,
aunque su causa no ha sido verificada.

Se conserva esta observación para futuras
comprobaciones de consistencia.

### Validación de fechas SECOP II

Esta consulta comprobará cuántos registros tienen fecha, cuántos carecen de ella y cuántos presentan fechas posteriores al día de la consulta.

In [0]:

import requests
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo

url = "https://www.datos.gov.co/resource/jbjy-vk9h.json"

# Fecha actual en Colombia
fecha_actual = datetime.now(
    ZoneInfo("America/Bogota")
).strftime("%Y-%m-%dT00:00:00")

consultas = {
    "Total de registros": None,
    "Registros con fecha": "fecha_de_firma IS NOT NULL",
    "Registros sin fecha": "fecha_de_firma IS NULL",
    "Fechas posteriores a hoy": (
        f"fecha_de_firma >= '{fecha_actual}'"
    )
}

resultados = []

for categoria, condicion in consultas.items():

    parametros = {
        "$select": "count(*) as total"
    }

    if condicion:
        parametros["$where"] = condicion

    respuesta = requests.get(
        url,
        params=parametros,
        timeout=120
    )

    respuesta.raise_for_status()

    resultados.append({
        "Categoría": categoria,
        "Registros": int(respuesta.json()[0]["total"])
    })

display(pd.DataFrame(resultados))

### Validación de fechas SECOP II

#### Resultados

| Indicador | Registros |
|---|---:|
| Total de registros | 5.687.382 |
| Con fecha de firma | 5.620.281 |
| Sin fecha de firma | 67.101 |
| Fechas posteriores al día de consulta | 0 |

**Completitud:** 98,82 %.

**Cobertura temporal observada:**
- Fecha mínima: 2015-06-11.
- Fecha máxima: 2026-11-08.

#### Hallazgos

1. La mayoría de los registros contiene una fecha de firma.
2. Se identificaron 67.101 registros sin fecha de firma.
3. No se encontraron fechas posteriores al día de consulta.
4. El campo utiliza el tipo de dato `calendar_date`.
5. La validez del formato no garantiza la coherencia
   de las fechas con el proceso contractual.

#### Conclusión

La validación exploratoria inicial evidencia una alta
completitud del campo `fecha_de_firma` en SECOP II.

No se identificaron fechas futuras y el intervalo
observado comprende desde junio de 2015 hasta
noviembre de 2026.

Queda pendiente evaluar la coherencia entre las fechas
de firma, inicio y finalización de los contratos.

Los registros sin fecha de firma no serán descartados
automáticamente del proyecto.

### Distribución anual de SECOP II

In [0]:


import requests
import pandas as pd

url = "https://www.datos.gov.co/resource/jbjy-vk9h.json"

parametros = {
    "$select": (
        "date_extract_y(fecha_de_firma) as anio, "
        "count(*) as total_registros"
    ),
    "$where": "fecha_de_firma IS NOT NULL",
    "$group": "date_extract_y(fecha_de_firma)",
    "$order": "anio ASC",
    "$limit": 100
}

respuesta = requests.get(
    url,
    params=parametros,
    timeout=120
)

respuesta.raise_for_status()

df_anual_secop = pd.DataFrame(respuesta.json())

df_anual_secop["anio"] = df_anual_secop["anio"].astype(int)
df_anual_secop["total_registros"] = (
    df_anual_secop["total_registros"].astype(int)
)

display(df_anual_secop)

print(
    "Total de registros distribuidos:",
    df_anual_secop["total_registros"].sum()
)

#### Verificación de consistencia

| Indicador | Registros |
|---|---:|
| Registros con fecha de firma | 5.620.281 |
| Registros distribuidos por año | 5.620.281 |
| **Diferencia** | **0** |

La suma de los registros agrupados por año coincide
exactamente con el conteo previo de registros que
contienen una fecha de firma.

Esta coincidencia confirma que la distribución anual
incluye la totalidad de los registros con fecha de firma
identificados durante la consulta.

Los 67.101 registros sin fecha de firma no aparecen
en la distribución anual, pero permanecen contabilizados
dentro del volumen total de la fuente.

#### Conclusión

SECOP II presenta una cobertura temporal desde 2015
hasta 2026, con un incremento importante del volumen
de registros en los periodos recientes.

La distribución anual presenta consistencia completa
con el conteo de registros que contienen fecha de firma.

Sin embargo, las variaciones anuales no deben interpretarse
automáticamente como crecimiento o disminución de la
contratación pública, debido a posibles diferencias
en la cobertura y actualización de la información.

Esta actividad permite caracterizar la disponibilidad
temporal de los datos sin establecer restricciones
sobre los registros que serán utilizados en el proyecto.